# MIST · ACDC Inference & Model Evaluation — RunPod

Loads one or more **already-trained** checkpoints from `model_pth/` on the runpod volume and runs a full
qualitative + quantitative evaluation suite on the ACDC test set, plus GPU inference-time benchmarking and
a training-curve comparison.

**This notebook does not train anything** — it only builds `MIST_CAM` (`lib/networks.py`), loads a `.pth`
checkpoint into it, and evaluates. It works on any checkpoint produced by `ACDC_train_runpod.ipynb` /
`ACDC_train_runpod_mamba.ipynb`, as long as the currently checked-out `lib/MIST.py` matches the architecture
that checkpoint was trained with (same Mamba/MHSA wiring in the decoder blocks — a checkpoint trained on a
different branch/architecture will fail to load with a key-mismatch error, which is reported per-model
rather than crashing the whole run).

**Contents**
1. Setup (installs)
2. Configuration + checkpoint discovery
3. `MODEL_REGISTRY` — pick which runs to evaluate
4. Quantitative evaluation (Dice / HD95 / Jaccard / ASD, per class, on the full ACDC test set)
5. Qualitative evaluation (overlay grids, error maps, confusion matrices)
6. Inference speed & model complexity (GPU latency/throughput, params, FLOPs)
7. Training curves comparison
8. Export (CSVs + PNGs under `eval_results/`)

**Run cells in order.** Section 3 (`MODEL_REGISTRY`) is the only cell you need to edit for a normal run.

In [ ]:
# Setup 1/2 — GPU check, protect torch/torchvision/numpy, install base dependencies.
# Mirrors the training notebooks' defensive install pattern: snapshot the pod's pristine
# torch/torchvision/numpy versions into a pip constraints file BEFORE installing anything else,
# so no later `pip install` here can silently swap them (that silent swap is what breaks
# `import timm` with confusing torch._dynamo/torchvision errors that have nothing to do with
# the actual package being installed).
import subprocess, sys
from importlib.metadata import version as pkg_version, PackageNotFoundError
import torch, torchvision

print(f'PyTorch     : {torch.__version__}')
print(f'torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU detected -- this notebook needs a CUDA GPU.'
print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def _v(name):
    try:
        return pkg_version(name)
    except PackageNotFoundError:
        return None

PROTECT = ['torch', 'torchvision', 'numpy']
CONSTRAINTS_PATH = '/tmp/mist_eval_constraints.txt'
with open(CONSTRAINTS_PATH, 'w') as f:
    for name in PROTECT:
        v = _v(name)
        if v:
            f.write(f'{name}=={v}\n')
before = {name: _v(name) for name in PROTECT}

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args), capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-600:]

pkgs = [
    'scipy>=1.14.0', 'timm==0.9.12', 'medpy', 'SimpleITK', 'seaborn',
    'segmentation-mask-overlay', 'thop', 'scikit-image', 'einops',
    'pandas', 'matplotlib', 'tqdm',
]
for spec in pkgs:
    code, out = pip('install', spec, '--quiet', '-c', CONSTRAINTS_PATH)
    print(f"  {'OK    ' if code == 0 else 'FAILED'}  {spec}")
    if code != 0:
        print(out)

after = {name: _v(name) for name in PROTECT}
changed = {k: (before[k], after[k]) for k in PROTECT if before[k] != after[k]}
if changed:
    raise RuntimeError(f'Base dependency install changed protected packages: {changed}')
print('\nBase dependencies installed; torch/torchvision/numpy unchanged.')

In [ ]:
# Setup 2/2 — mamba-ssm + causal-conv1d.
# The current lib/MIST.py builds DirectionalMambaSSM (lib/mamba_block.py) into the decoder, which
# requires mamba_ssm even just to construct the model for inference. Skips the (slow) CUDA build
# entirely if a working mamba_ssm is already importable in this environment.
import os

def _mamba_ready():
    try:
        import torch
        from mamba_ssm import Mamba
        core = Mamba(d_model=16, d_state=16, d_conv=4, expand=2).cuda()
        y = core(torch.randn(1, 32, 16, device='cuda'))
        del core, y
        torch.cuda.empty_cache()
        return True
    except Exception:
        return False

if _mamba_ready():
    print('mamba_ssm already installed and working -- skipping build.')
else:
    import subprocess, sys
    from importlib.metadata import version as pkg_version, PackageNotFoundError

    PROTECT = ['torch', 'torchvision', 'numpy']

    def _v(name):
        try:
            return pkg_version(name)
        except PackageNotFoundError:
            return None

    before = {name: _v(name) for name in PROTECT}
    env = os.environ.copy()
    env.setdefault('MAX_JOBS', '4')

    def pip_build(*args, label):
        cmd = [sys.executable, '-m', 'pip', 'install'] + list(args)
        r = subprocess.run(cmd, capture_output=True, text=True, env=env)
        print(r.stdout[-2000:])
        if r.returncode != 0:
            print(r.stderr[-2000:])
            raise RuntimeError(f'{label} install failed (see log above).')
        print(f'{label} installed OK')

    pip_build('packaging', 'ninja', label='build tooling')
    pip_build('causal-conv1d>=1.2.0', '--no-build-isolation', '-c', '/tmp/mist_eval_constraints.txt', label='causal-conv1d')
    pip_build('mamba-ssm', '--no-build-isolation', '-c', '/tmp/mist_eval_constraints.txt', label='mamba-ssm')

    after = {name: _v(name) for name in PROTECT}
    changed = {k: (before[k], after[k]) for k in PROTECT if before[k] != after[k]}
    if changed:
        raise RuntimeError(f'mamba-ssm install changed protected packages: {changed}. Restart the kernel and re-run.')
    assert _mamba_ready(), 'mamba_ssm still not importable/working after install.'
    print('mamba_ssm installed and verified working.')

In [ ]:
# Common imports used throughout this notebook
import os, sys, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image as PILImage
from tqdm import tqdm
import torch

print('Common imports OK')

In [ ]:
REPO_DIR = '/workspace/MIST'   # change if your repo/volume lives elsewhere
if not os.path.isdir(REPO_DIR):
    REPO_DIR = os.getcwd()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

from lib.networks import MIST_CAM
from utils.dataset_ACDC import ACDCdataset
from utils.utils import calculate_metric_percase

print('Project modules imported OK')

## Configuration

In [ ]:
MODEL_PTH_ROOT = './model_pth'        # root folder with one subfolder per training run (relative to REPO_DIR)
DATA_DIR       = './data/ACDC'
LIST_DIR       = './data/ACDC/lists_ACDC'
TEST_DIR       = './data/ACDC/test'

CLASS_NAMES     = ['RV', 'Myo', 'LV']    # class indices 1,2,3 -- background (0) excluded from all reported metrics
EVAL_OUTPUT_DIR = './eval_results'
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

# Fallback values used whenever a MODEL_REGISTRY entry and the run's own config.txt both omit a field
DEFAULTS = dict(
    num_classes=len(CLASS_NAMES) + 1,
    img_size=256,
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
    checkpoint='best.pth',
)

print(f'Model root : {os.path.abspath(MODEL_PTH_ROOT)}')
print(f'Test dir   : {os.path.abspath(TEST_DIR)}')

In [ ]:
# Discover run folders under MODEL_PTH_ROOT and read each one's config.txt (written by the training
# notebooks) so MODEL_REGISTRY entries below can leave img_size/num_classes/etc. unset and inherit them.

def parse_config_txt(path):
    cfg = {}
    if not os.path.exists(path):
        return cfg
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if '=' not in line:
                continue
            k, v = line.split('=', 1)
            cfg[k.strip()] = v.strip()
    return cfg


def discover_runs(root):
    runs = []
    if not os.path.isdir(root):
        print(f'WARNING: {root} does not exist (yet) -- check MODEL_PTH_ROOT / that the volume is mounted.')
        return runs
    for entry in sorted(os.listdir(root)):
        run_dir = os.path.join(root, entry)
        if not os.path.isdir(run_dir):
            continue
        cfg = parse_config_txt(os.path.join(run_dir, 'config.txt'))
        runs.append(dict(
            run_dir=entry,
            has_best=os.path.exists(os.path.join(run_dir, 'best.pth')),
            has_last=os.path.exists(os.path.join(run_dir, 'last.pth')),
            has_curves=os.path.exists(os.path.join(run_dir, 'training_curves.png')),
            config=cfg,
        ))
    return runs


discovered = discover_runs(MODEL_PTH_ROOT)

print(f'{"run_dir":<42} {"best.pth":<9} {"last.pth":<9} {"curves":<7} {"img_size":<9} {"epochs":<7}')
for r in discovered:
    c = r['config']
    print(f"{r['run_dir']:<42} {str(r['has_best']):<9} {str(r['has_last']):<9} "
          f"{str(r['has_curves']):<7} {c.get('img_size', '?'):<9} {c.get('epochs', '?'):<7}")

print(f'\n{len(discovered)} run folder(s) found under {MODEL_PTH_ROOT}')
print('Copy the run_dir names you want into MODEL_REGISTRY below.')

## Choose which models to evaluate

Only the runs listed in `MODEL_REGISTRY` below get loaded and evaluated — everything else in `model_pth/`
is ignored. Copy `run_dir` values from the discovery table above. Every field except `run_dir` is optional:
unset fields are read from that run's `config.txt`, then from `DEFAULTS`.

```python
dict(
    name='...',                 # display name used in tables/plots/legends (defaults to run_dir)
    run_dir='...',              # required -- folder name under MODEL_PTH_ROOT
    checkpoint='best.pth',      # or 'last.pth' (a resumable dict; model_state_dict is extracted automatically)
    num_classes=4,
    img_size=256,
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
)
```

In [ ]:
# Only the runs listed here get loaded and evaluated below.
MODEL_REGISTRY = [
    # dict(name='MIST_CAM Mamba (run143022)', run_dir='MIST_CAM_mamba_256_run143022'),
    # dict(name='MIST_CAM Mamba (run091500)', run_dir='MIST_CAM_mamba_256_run091500', checkpoint='last.pth'),
]

assert MODEL_REGISTRY, (
    'MODEL_REGISTRY is empty -- copy run_dir names from the discovery table above and add at least one entry.')

print(f'{len(MODEL_REGISTRY)} model(s) registered for evaluation:')
for m in MODEL_REGISTRY:
    print(' -', m.get('name', m['run_dir']))

In [ ]:
def resolve_entry(entry):
    run_dir = os.path.join(MODEL_PTH_ROOT, entry['run_dir'])
    cfg = parse_config_txt(os.path.join(run_dir, 'config.txt'))

    def field(key, cast=str):
        if entry.get(key) is not None:
            return entry[key]
        if key in cfg:
            return cast(cfg[key])
        return DEFAULTS[key]

    return dict(
        name=entry.get('name', entry['run_dir']),
        run_dir=run_dir,
        checkpoint=entry.get('checkpoint', DEFAULTS['checkpoint']),
        num_classes=field('num_classes', int),
        img_size=field('img_size', int),
        model_scale=field('model_scale', str),
        decoder_aggregation=field('decoder_aggregation', str),
        interpolation=field('interpolation', str),
    )


def build_model(resolved):
    return MIST_CAM(
        n_class=resolved['num_classes'],
        img_size_s1=(resolved['img_size'], resolved['img_size']),
        img_size_s2=(224, 224),
        model_scale=resolved['model_scale'],
        decoder_aggregation=resolved['decoder_aggregation'],
        interpolation=resolved['interpolation'],
    ).cuda()


def load_checkpoint(net, ckpt_path):
    state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    # torch.compile wraps keys with a `_orig_mod.` prefix -- strip it for plain model loading
    if any(k.startswith('_orig_mod.') for k in state):
        state = {k.replace('_orig_mod.', '', 1): v for k, v in state.items()}
    net.load_state_dict(state)
    return net

In [ ]:
# Build + load every registered model. A checkpoint that doesn't match the currently checked-out
# architecture (e.g. trained on a different branch) fails here with a clear message instead of
# silently producing garbage predictions later.
MODELS = {}   # name -> {'net', 'resolved', 'n_params_m'}

for entry in MODEL_REGISTRY:
    resolved = resolve_entry(entry)
    assert resolved['num_classes'] == len(CLASS_NAMES) + 1, (
        f"{resolved['name']}: num_classes={resolved['num_classes']} doesn't match "
        f"CLASS_NAMES={CLASS_NAMES} (expected {len(CLASS_NAMES) + 1}) -- adjust CLASS_NAMES or this entry.")
    ckpt_path = os.path.join(resolved['run_dir'], resolved['checkpoint'])
    print(f"Loading '{resolved['name']}' from {ckpt_path} ...")
    try:
        net = build_model(resolved)
        load_checkpoint(net, ckpt_path)
        net.eval()
        n_params_m = sum(p.numel() for p in net.parameters()) / 1e6
        MODELS[resolved['name']] = dict(net=net, resolved=resolved, n_params_m=n_params_m)
        print(f"  OK -- params={n_params_m:.2f}M  img_size={resolved['img_size']}  num_classes={resolved['num_classes']}")
    except Exception as e:
        print(f"  FAILED to load '{resolved['name']}': {e}")

assert MODELS, 'No models loaded successfully -- check the errors above.'
print(f'\n{len(MODELS)}/{len(MODEL_REGISTRY)} model(s) loaded successfully.')
model_names = list(MODELS.keys())

## Quantitative evaluation

Runs every loaded model over the full ACDC test set and reports the paper's metric suite per class:
**Dice, HD95, Jaccard, ASD** (background excluded), using the same `calculate_metric_percase` (medpy-based)
that `test_ACDC.py` / `inference_ACDC.py` use, so numbers are directly comparable to prior reported results.

In [ ]:
from scipy.ndimage import zoom as zoom_fn
from torch.utils.data import DataLoader

db_test    = ACDCdataset(base_dir=TEST_DIR, list_dir=LIST_DIR, split='test')
testloader = DataLoader(db_test, batch_size=1, shuffle=False)
print(f'Test set: {len(db_test)} item(s)')


def run_case_inference(net, image, img_size):
    """image: (H,W) or (S,H,W) numpy array -> full-resolution int prediction array, same shape."""
    squeeze = image.ndim == 2
    vol = image[None, ...] if squeeze else image
    pred = np.zeros(vol.shape, dtype=np.int64)
    for si in range(vol.shape[0]):
        slc = vol[si]
        h, w = slc.shape
        if h != img_size or w != img_size:
            slc = zoom_fn(slc, (img_size / h, img_size / w), order=3)
        t = torch.from_numpy(slc).unsqueeze(0).unsqueeze(0).float().cuda()
        with torch.no_grad():
            out = sum(net(t))
            out = torch.argmax(torch.softmax(out, dim=1), dim=1).squeeze(0).cpu().numpy()
        if h != img_size or w != img_size:
            out = zoom_fn(out, (h / img_size, w / img_size), order=0)
        pred[si] = out
    return pred[0] if squeeze else pred

In [ ]:
# One evaluation pass per model: computes the official per-class metrics AND keeps the full
# prediction volumes (used later for confusion matrices and qualitative overlays) -- avoids
# running inference twice.
RESULTS = {}   # name -> dict(metrics=(N, n_classes-1, 4) array, case_names=[...], predictions={case: array}, confmat)

for name, m in MODELS.items():
    net, img_size, num_classes = m['net'], m['resolved']['img_size'], m['resolved']['num_classes']
    print(f'Evaluating {name} on {len(db_test)} test case(s)...')

    metric_list_all, case_names, predictions = [], [], {}
    confmat = np.zeros((num_classes, num_classes), dtype=np.int64)

    for sampled_batch in tqdm(testloader, desc=name):
        image = sampled_batch['image'].squeeze(0).cpu().numpy()
        label = sampled_batch['label'].squeeze(0).cpu().numpy()
        case_name = sampled_batch['case_name'][0]

        pred = run_case_inference(net, image, img_size)

        metric_list_all.append([calculate_metric_percase(pred.copy() == c, label.copy() == c)
                                for c in range(1, num_classes)])
        case_names.append(case_name)
        predictions[case_name] = pred

        lbl_i = label.astype(np.int64).ravel()
        pred_i = pred.astype(np.int64).ravel()
        confmat += np.bincount(num_classes * lbl_i + pred_i, minlength=num_classes ** 2).reshape(num_classes, num_classes)

    metrics = np.array(metric_list_all)   # (N, num_classes-1, 4) -- last axis: dice, hd95, jaccard, asd
    RESULTS[name] = dict(metrics=metrics, case_names=case_names, predictions=predictions, confmat=confmat)
    print(f'  Mean Dice: {metrics[:, :, 0].mean() * 100:.2f}%')

print('\nAll models evaluated.')

In [ ]:
# Per-model, per-class summary table
rows = []
for name, res in RESULTS.items():
    metrics = res['metrics']
    row = {'model': name, 'n_cases': metrics.shape[0]}
    for ci, cname in enumerate(CLASS_NAMES):
        row[f'{cname}_Dice']    = metrics[:, ci, 0].mean() * 100
        row[f'{cname}_HD95']    = metrics[:, ci, 1].mean()
        row[f'{cname}_Jaccard'] = metrics[:, ci, 2].mean() * 100
        row[f'{cname}_ASD']     = metrics[:, ci, 3].mean()
    row['Mean_Dice']    = metrics[:, :, 0].mean() * 100
    row['Mean_HD95']    = metrics[:, :, 1].mean()
    row['Mean_Jaccard'] = metrics[:, :, 2].mean() * 100
    row['Mean_ASD']     = metrics[:, :, 3].mean()
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('model')
summary_df.to_csv(os.path.join(EVAL_OUTPUT_DIR, 'summary_metrics.csv'))
print(f"Saved: {os.path.join(EVAL_OUTPUT_DIR, 'summary_metrics.csv')}")
summary_df.round(2)

In [ ]:
# Mean Dice per class, grouped by model
x = np.arange(len(CLASS_NAMES))
width = 0.8 / max(len(model_names), 1)

fig, ax = plt.subplots(figsize=(8, 5))
for i, name in enumerate(model_names):
    means = [RESULTS[name]['metrics'][:, ci, 0].mean() * 100 for ci in range(len(CLASS_NAMES))]
    ax.bar(x + i * width - 0.4 + width / 2, means, width, label=name)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Dice (%)')
ax.set_ylim(0, 105)
ax.set_title('Mean Dice per class — model comparison')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'compare_dice_per_class.png'), dpi=150)
plt.show()

In [ ]:
# HD95 / Jaccard / ASD per class, grouped by model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_specs = [('HD95', 1, 'px', False), ('Jaccard', 2, '%', True), ('ASD', 3, 'px', False)]

for ax, (mname, mi, unit, as_pct) in zip(axes, metric_specs):
    for i, name in enumerate(model_names):
        vals = RESULTS[name]['metrics'][:, :, mi]
        means = vals.mean(axis=0) * (100 if as_pct else 1)
        ax.bar(x + i * width - 0.4 + width / 2, means, width, label=name)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylabel(f'{mname} ({unit})')
    ax.set_title(f'Mean {mname} per class')
    ax.grid(True, alpha=0.3, axis='y')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'compare_other_metrics.png'), dpi=150)
plt.show()

In [ ]:
# Dice distribution across test cases, per class, per model
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(5 * len(CLASS_NAMES), 5), sharey=True)
axes = np.atleast_1d(axes)
colors = plt.cm.tab10(np.linspace(0, 1, len(model_names)))

for ci, cname in enumerate(CLASS_NAMES):
    ax = axes[ci]
    data = [RESULTS[name]['metrics'][:, ci, 0] * 100 for name in model_names]
    bp = ax.boxplot(data, showmeans=True, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticks(range(1, len(model_names) + 1))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
    ax.set_title(f'{cname} Dice distribution')
    ax.set_ylabel('Dice (%)')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'dice_distributions.png'), dpi=150)
plt.show()

In [ ]:
# Pixel-level confusion matrix per model (row-normalized: recall per true class)
fig, axes = plt.subplots(1, len(RESULTS), figsize=(5 * len(RESULTS), 4.5))
axes = np.atleast_1d(axes)
labels = ['BG'] + CLASS_NAMES

for ax, (name, res) in zip(axes, RESULTS.items()):
    cm = res['confmat'].astype(np.float64)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
    ax.set_title(name, fontsize=9)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center',
                    color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

## Qualitative evaluation

In [ ]:
# Pick representative cases (best / median / worst mean Dice) using the first registered model as
# reference, so every model is visualized on the same cases for a fair side-by-side comparison.
N_PER_BUCKET = 2

reference_model  = model_names[0]
ref_metrics      = RESULTS[reference_model]['metrics']
ref_case_names   = RESULTS[reference_model]['case_names']
mean_dice_per_case = ref_metrics[:, :, 0].mean(axis=1)
order = np.argsort(mean_dice_per_case)

worst_idx    = order[:N_PER_BUCKET]
best_idx     = order[-N_PER_BUCKET:]
median_start = len(order) // 2 - N_PER_BUCKET // 2
median_idx   = order[median_start:median_start + N_PER_BUCKET]

QUALITATIVE_CASES = []
for bucket_name, idxs in [('best', best_idx), ('median', median_idx), ('worst', worst_idx)]:
    for idx in idxs:
        QUALITATIVE_CASES.append((bucket_name, ref_case_names[idx]))

print(f'Reference model for case selection: {reference_model}')
for bucket, case in QUALITATIVE_CASES:
    print(f'  [{bucket:6s}] {case}')

In [ ]:
def pick_slice(label):
    """Index of the slice with the largest foreground area, or None for an already-2D case."""
    if label.ndim == 2:
        return None
    areas = (label > 0).reshape(label.shape[0], -1).sum(axis=1)
    return int(np.argmax(areas))


def get_2d(arr, slice_idx):
    return arr if arr.ndim == 2 else arr[slice_idx]


SEG_CMAP = mcolors.ListedColormap(['black', 'red', 'gold', 'deepskyblue'])  # BG, RV, Myo, LV

n_cols = 2 + len(model_names)
fig, axes = plt.subplots(len(QUALITATIVE_CASES), n_cols,
                         figsize=(3 * n_cols, 3 * len(QUALITATIVE_CASES)), squeeze=False)

for row, (bucket, case) in enumerate(QUALITATIVE_CASES):
    data = np.load(os.path.join(TEST_DIR, case))
    img, lbl = data['img'], data['label']
    sl = pick_slice(lbl)
    img2d, lbl2d = get_2d(img, sl), get_2d(lbl, sl)

    ax = axes[row, 0]
    ax.imshow(img2d, cmap='gray')
    if row == 0:
        ax.set_title('Image')
    ax.set_ylabel(f'[{bucket}]\n{case}', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

    ax = axes[row, 1]
    ax.imshow(img2d, cmap='gray')
    ax.imshow(lbl2d, cmap=SEG_CMAP, alpha=0.5, vmin=0, vmax=len(CLASS_NAMES))
    if row == 0:
        ax.set_title('Ground truth')
    ax.set_xticks([]); ax.set_yticks([])

    for col, name in enumerate(model_names):
        pred2d = get_2d(RESULTS[name]['predictions'][case], sl)
        dice_here = np.mean([calculate_metric_percase(pred2d.copy() == c, lbl2d.copy() == c)[0]
                             for c in range(1, len(CLASS_NAMES) + 1)])
        ax = axes[row, 2 + col]
        ax.imshow(img2d, cmap='gray')
        ax.imshow(pred2d, cmap=SEG_CMAP, alpha=0.5, vmin=0, vmax=len(CLASS_NAMES))
        ax.set_title(f'{name}\nDice={dice_here * 100:.1f}%' if row == 0 else f'Dice={dice_here * 100:.1f}%', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'qualitative_overlays.png'), dpi=150)
plt.show()

In [ ]:
# Error maps: green = correctly predicted foreground, red = any mismatch (wrong class or missed/extra)
fig, axes = plt.subplots(len(QUALITATIVE_CASES), len(model_names),
                         figsize=(3.5 * len(model_names), 3 * len(QUALITATIVE_CASES)), squeeze=False)

for row, (bucket, case) in enumerate(QUALITATIVE_CASES):
    data = np.load(os.path.join(TEST_DIR, case))
    lbl = data['label']
    sl = pick_slice(lbl)
    lbl2d = get_2d(lbl, sl)

    for col, name in enumerate(model_names):
        pred2d = get_2d(RESULTS[name]['predictions'][case], sl)
        err = np.zeros((*lbl2d.shape, 3))
        err[(pred2d == lbl2d) & (lbl2d > 0)] = [0.0, 0.7, 0.0]
        err[pred2d != lbl2d]                 = [0.9, 0.0, 0.0]

        ax = axes[row, col]
        ax.imshow(err)
        if row == 0:
            ax.set_title(name, fontsize=9)
        if col == 0:
            ax.set_ylabel(f'[{bucket}]\n{case}', fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'error_maps.png'), dpi=150)
plt.show()

## Inference speed & model complexity

In [ ]:
N_WARMUP = 10
N_TIMED  = 50
BENCH_BATCH_SIZES = [1, 8]


def benchmark_model(net, img_size, batch_size):
    x = torch.randn(batch_size, 1, img_size, img_size, device='cuda')
    net.eval()
    with torch.no_grad():
        for _ in range(N_WARMUP):
            net(x)
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(N_TIMED):
            net(x)
        end.record()
        torch.cuda.synchronize()
    per_call_ms = start.elapsed_time(end) / N_TIMED
    throughput  = batch_size * 1000.0 / per_call_ms
    return per_call_ms, throughput


BENCH_RESULTS = {}
for name, m in MODELS.items():
    net, img_size = m['net'], m['resolved']['img_size']
    BENCH_RESULTS[name] = {}
    for bs in BENCH_BATCH_SIZES:
        try:
            latency_ms, throughput = benchmark_model(net, img_size, bs)
            BENCH_RESULTS[name][bs] = dict(latency_ms=latency_ms, throughput=throughput)
            print(f'{name:<35} batch={bs:<3} latency={latency_ms:7.2f} ms  throughput={throughput:8.1f} img/s')
        except RuntimeError as e:
            print(f'{name:<35} batch={bs:<3} FAILED: {e}')
    torch.cuda.empty_cache()

In [ ]:
from thop import profile

COMPLEXITY = {}
for name, m in MODELS.items():
    net, img_size = m['net'], m['resolved']['img_size']
    x = torch.randn(1, 1, img_size, img_size, device='cuda')
    try:
        macs, params = profile(net, inputs=(x,), verbose=False)
        COMPLEXITY[name] = dict(gflops=macs / 1e9, params_m=params / 1e6)
    except Exception as e:
        print(f'FLOPs profiling failed for {name}: {e}')
        COMPLEXITY[name] = dict(gflops=float('nan'), params_m=m['n_params_m'])

for name, c in COMPLEXITY.items():
    print(f"{name:<35} params={c['params_m']:7.2f}M  GFLOPs={c['gflops']:8.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

lat1 = [BENCH_RESULTS[n][BENCH_BATCH_SIZES[0]]['latency_ms'] for n in model_names]
axes[0, 0].bar(model_names, lat1, color='steelblue')
axes[0, 0].set_title(f'Latency @ batch={BENCH_BATCH_SIZES[0]}'); axes[0, 0].set_ylabel('ms/call')

thr1 = [BENCH_RESULTS[n][BENCH_BATCH_SIZES[0]]['throughput'] for n in model_names]
axes[0, 1].bar(model_names, thr1, color='seagreen')
axes[0, 1].set_title(f'Throughput @ batch={BENCH_BATCH_SIZES[0]}'); axes[0, 1].set_ylabel('images/sec')

params = [COMPLEXITY[n]['params_m'] for n in model_names]
axes[1, 0].bar(model_names, params, color='indianred')
axes[1, 0].set_title('Parameters'); axes[1, 0].set_ylabel('Millions')

gflops = [COMPLEXITY[n]['gflops'] for n in model_names]
axes[1, 1].bar(model_names, gflops, color='goldenrod')
axes[1, 1].set_title('Compute (single-image forward)'); axes[1, 1].set_ylabel('GFLOPs')

for ax in axes.ravel():
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'compare_efficiency.png'), dpi=150)
plt.show()

In [ ]:
# Accuracy vs. speed trade-off — bubble size = parameter count
fig, ax = plt.subplots(figsize=(7, 6))
for name in model_names:
    dice   = summary_df.loc[name, 'Mean_Dice']
    lat    = BENCH_RESULTS[name][BENCH_BATCH_SIZES[0]]['latency_ms']
    params = COMPLEXITY[name]['params_m']
    ax.scatter(lat, dice, s=params * 15, alpha=0.6, edgecolor='black')
    ax.annotate(name, (lat, dice), fontsize=8, xytext=(5, 5), textcoords='offset points')
ax.set_xlabel(f'Inference latency @ batch={BENCH_BATCH_SIZES[0]} (ms)')
ax.set_ylabel('Mean Dice (%)')
ax.set_title('Accuracy vs. speed (bubble size = parameter count)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'accuracy_vs_speed.png'), dpi=150)
plt.show()

## Training curves

The training notebooks only persist per-epoch loss/Dice history as a rendered `training_curves.png`
image per run (the raw numbers aren't saved to disk), so full curves from different runs can't be
overlaid onto one shared-axis line plot. This section shows each run's saved image side by side, plus
a bar-chart comparison of the two numbers that *are* persisted in `last.pth`: best validation Dice and
epochs completed.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 4))
axes = np.atleast_1d(axes)

for ax, (name, m) in zip(axes, MODELS.items()):
    curve_path = os.path.join(m['resolved']['run_dir'], 'training_curves.png')
    if os.path.exists(curve_path):
        ax.imshow(PILImage.open(curve_path))
    else:
        ax.text(0.5, 0.5, 'training_curves.png\nnot found', ha='center', va='center')
    ax.set_title(name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'training_curves_grid.png'), dpi=150)
plt.show()

In [ ]:
TRAIN_SUMMARY = {}
for name, m in MODELS.items():
    last_path = os.path.join(m['resolved']['run_dir'], 'last.pth')
    if os.path.exists(last_path):
        ckpt = torch.load(last_path, map_location='cpu', weights_only=False)
        TRAIN_SUMMARY[name] = dict(
            best_val_dice=ckpt.get('best_dcs', float('nan')) * 100,
            epochs_trained=(ckpt['epoch'] + 1) if 'epoch' in ckpt else float('nan'),
        )
    else:
        TRAIN_SUMMARY[name] = dict(best_val_dice=float('nan'), epochs_trained=float('nan'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(model_names, [TRAIN_SUMMARY[n]['best_val_dice'] for n in model_names], color='seagreen')
axes[0].set_title('Best validation Dice (from last.pth)'); axes[0].set_ylabel('Dice (%)')
axes[1].bar(model_names, [TRAIN_SUMMARY[n]['epochs_trained'] for n in model_names], color='slateblue')
axes[1].set_title('Epochs trained (from last.pth)'); axes[1].set_ylabel('Epochs')

for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUTPUT_DIR, 'training_summary.png'), dpi=150)
plt.show()

## Export

In [ ]:
detailed_rows = []
for name, res in RESULTS.items():
    for ci_idx, case in enumerate(res['case_names']):
        row = {'model': name, 'case': case}
        for ci, cname in enumerate(CLASS_NAMES):
            row[f'{cname}_Dice']    = res['metrics'][ci_idx, ci, 0] * 100
            row[f'{cname}_HD95']    = res['metrics'][ci_idx, ci, 1]
            row[f'{cname}_Jaccard'] = res['metrics'][ci_idx, ci, 2] * 100
            row[f'{cname}_ASD']     = res['metrics'][ci_idx, ci, 3]
        detailed_rows.append(row)

detailed_df = pd.DataFrame(detailed_rows)
detailed_csv = os.path.join(EVAL_OUTPUT_DIR, 'per_case_metrics.csv')
detailed_df.to_csv(detailed_csv, index=False)
print(f'Saved: {detailed_csv}')

print(f'\nAll figures and CSVs saved under: {os.path.abspath(EVAL_OUTPUT_DIR)}')
for f in sorted(os.listdir(EVAL_OUTPUT_DIR)):
    print(' -', f)